# Task 3 — MAE-aligned hurdle follow-up

Extends the friend's `build_model_v5/v6` procedure with the same 48 inputs, parameters and five development folds. Refit the source baseline and compare challengers against that refit. Published scores are shown separately because the original gate OOF caches are unavailable. This experiment does not fit on holdout or Test, and does not assemble a Kaggle submission.

## 1. Setup & Raw Train

In [1]:
from pathlib import Path
import sys, json, hashlib
from importlib.metadata import version
import numpy as np
import pandas as pd
from IPython.display import display
ROOT = next((candidate for base in (Path.cwd(), *Path.cwd().parents)
             for candidate in (base, base / "CPE342-Kaggle")
             if (candidate / "Task3" / "models" / "hurdle_followup.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open CPE342-Kaggle or a folder inside it")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from Task3.models import hurdle_followup as experiment
TASK = ROOT / "Task3"
RAW_PATH = TASK / "data" / "raw" / "train.csv"
raw = pd.read_csv(RAW_PATH)
print("Train shape:", raw.shape)


Train shape: (104000, 34)


## 2. Reproduce Features & Development Folds

In [2]:
dev, features, folds, dev_pos, hold_pos = experiment.prepare_development(raw)
assert len(features) == 48 and len(dev) == 83200 and len(hold_pos) == 20800
feature_manifest = pd.DataFrame({"feature": features,
    "source": ["raw" if c in raw.columns else "friend_rowwise_formula" for c in features]})
display(feature_manifest)
print("Five development validation folds:", [len(check) for _, check in folds])
print("Holdout rows reserved and not scored in this experiment:", len(hold_pos))


,feature,source
0,friend_count,raw
1,social_interactions,raw
2,guild_membership,raw
3,event_participation_rate,raw
4,daily_login_streak,raw
5,avg_session_length,raw
6,sessions_per_week,raw
7,total_playtime_hours,raw
8,days_since_last_login,raw
9,achievement_count,raw


Five development validation folds: [16640, 16640, 16640, 16640, 16640]
Holdout rows reserved and not scored in this experiment: 20800


## 3. Fixed Experiment Settings

In [3]:
RUN_NAME = "hurdle_followup_v1"
OUT = TASK / "models" / "results" / RUN_NAME
MIN_MEAN_GAIN_THB = 10.0
MIN_FOLD_WINS = 4
settings = {
    "run": RUN_NAME, "seed": experiment.SEED,
    "source_revision": "32dc9afa34e7b8fb7cff0913241b591b7c695313",
    "train_sha256": hashlib.sha256(RAW_PATH.read_bytes()).hexdigest(),
    "implementation_sha256": hashlib.sha256(Path(experiment.__file__).read_bytes()).hexdigest(),
    "feature_sha256": hashlib.sha256(pd.util.hash_pandas_object(dev[features], index=True).to_numpy().tobytes()).hexdigest(),
    "development_positions_sha256": hashlib.sha256(dev_pos.astype("int64").tobytes()).hexdigest(),
    "features": features, "nominal": experiment.NOMINAL,
    "cat_params": experiment.CAT_PARAMS,
    "positive_quantiles": experiment.QUANTILES.tolist(),
    "candidates": experiment.CANDIDATES,
    "baseline": {"gate_threshold": .5, "gate_weight": .8, "continuous_cat_weight": .4},
    "baseline_reference_tolerance_thb": experiment.REFERENCE_SANITY_TOLERANCE_THB,
    "decision": {"minimum_mean_gain_thb": MIN_MEAN_GAIN_THB, "minimum_fold_wins": MIN_FOLD_WINS},
    "versions": {name: version(name) for name in ["numpy", "pandas", "scikit-learn", "catboost", "lightgbm"]},
}
OUT.mkdir(parents=True, exist_ok=True)
settings_path = OUT / "settings.json"
if settings_path.exists():
    assert json.loads(settings_path.read_text()) == settings, "Configuration changed: use a new RUN_NAME"
else:
    assert not any(OUT.iterdir()), "Nonempty run folder without settings"
    settings_path.write_text(json.dumps(settings, indent=2) + "\n")
feature_manifest.to_csv(OUT / "feature_manifest.csv", index=False)
fold_assignments = pd.concat([
    pd.DataFrame({"fold": i, "row_pos": check, "raw_index": dev.iloc[check].raw_index.to_numpy(),
                  "player_id": dev.iloc[check].player_id.to_numpy()})
    for i, (_, check) in enumerate(folds, 1)], ignore_index=True)
assert fold_assignments.row_pos.is_unique and len(fold_assignments) == len(dev)
fold_assignments.to_parquet(OUT / "fold_assignments.parquet", index=False)
print("Artifacts:", OUT)


Artifacts: /Users/meng/dev/MyUniversity/CPE342-MachineLearning/CPE342-Kaggle/Task3/models/results/hurdle_followup_v1


## 4. Train & Collect OOF Predictions

Six component fits per fold: the original gate, original positive log-MAE amount, original continuous CatBoost/LightGBM, direct all-row log-MAE, and a positive log-MultiQuantile model. The quantile model also supplies a positive-median control to separate representation changes from the conditional-quantile decision rule. Each component has a resume checkpoint. A baseline deviation over ฿10 from a published fold stops execution for investigation; smaller deviations are reported, and challenger gains always use the local refit as reference.

In [4]:
oof_parts = []
for fold_number in range(1, 6):
    part = experiment.run_fold(dev, features, folds, fold_number, OUT)
    oof_parts.append(part)
    display(pd.DataFrame([{"model": name, "fold": fold_number,
                          "MAE_thb": np.abs(part[name] - part.true).mean()}
                         for name in experiment.CANDIDATES]).round(3))
oof = pd.concat(oof_parts, ignore_index=True).sort_values("row_pos").reset_index(drop=True)
assert np.array_equal(oof.row_pos, np.arange(len(dev)))
assert np.array_equal(oof.player_id, dev.player_id)
assert np.array_equal(oof.true, dev.spending_30d)
assert np.isfinite(oof[experiment.CANDIDATES].to_numpy()).all()
assert oof[experiment.CANDIDATES].ge(0).all().all()
experiment.atomic_parquet(oof, OUT / "oof_predictions.parquet")


fold 1/5 baseline MAE: 2742.1382


,model,fold,MAE_thb
0,friend_log_mixture,1,2742.138
1,friend_log_gate_only,1,2748.661
2,direct_log_mae,1,3029.838
3,quantile_median_gate,1,2734.547
4,conditional_quantile_gate,1,2713.168
5,conditional_quantile_mixture,1,2720.536


fold 2/5 baseline MAE: 2885.2838


,model,fold,MAE_thb
0,friend_log_mixture,2,2885.284
1,friend_log_gate_only,2,2889.775
2,direct_log_mae,2,3002.993
3,quantile_median_gate,2,2892.980
4,conditional_quantile_gate,2,2876.028
5,conditional_quantile_mixture,2,2879.577


fold 3/5 baseline MAE: 2712.0340


,model,fold,MAE_thb
0,friend_log_mixture,3,2712.034
1,friend_log_gate_only,3,2713.312
2,direct_log_mae,3,2861.475
3,quantile_median_gate,3,2707.277
4,conditional_quantile_gate,3,2692.120
5,conditional_quantile_mixture,3,2703.593


fold 4/5 baseline MAE: 2848.2333


,model,fold,MAE_thb
0,friend_log_mixture,4,2848.233
1,friend_log_gate_only,4,2842.630
2,direct_log_mae,4,3218.286
3,quantile_median_gate,4,2833.846
4,conditional_quantile_gate,4,2819.434
5,conditional_quantile_mixture,4,2837.002


fold 5/5 baseline MAE: 2884.9548


,model,fold,MAE_thb
0,friend_log_mixture,5,2884.955
1,friend_log_gate_only,5,2890.450
2,direct_log_mae,5,3160.434
3,quantile_median_gate,5,2872.603
4,conditional_quantile_gate,5,2848.330
5,conditional_quantile_mixture,5,2856.920


## 5. Paired Fold Scores & Error Slices

In [5]:
fold_scores, ranking, error_slices = experiment.score_experiment(oof)
fold_scores.to_csv(OUT / "fold_scores.csv", index=False)
ranking.to_csv(OUT / "model_ranking.csv", index=False)
error_slices.to_csv(OUT / "error_slices.csv", index=False)
reconstructed_mean = ranking.loc[ranking.model.eq("friend_log_mixture"), "mean_MAE_thb"].item()
assert abs(reconstructed_mean - experiment.REFERENCE_MEAN_MAE) < experiment.REFERENCE_SANITY_TOLERANCE_THB
print("Published friend mean:", experiment.REFERENCE_MEAN_MAE, "| local-minus-published:", reconstructed_mean - experiment.REFERENCE_MEAN_MAE)
print("Friend baseline mean reconstructed:", reconstructed_mean)
display(ranking.round(4))
display(fold_scores.pivot(index="fold", columns="model", values="MAE_thb").round(2))


Published friend mean: 2814.9680123616345 | local-minus-published: -0.4391903223099689
Friend baseline mean reconstructed: 2814.5288220393245


,model,mean_MAE_thb,std_MAE_thb,delta_vs_friend_thb,folds_better,folds_worse
0,conditional_quantile_gate,2789.8159,82.3908,-24.7129,5,0
1,conditional_quantile_mixture,2799.5255,81.4699,-15.0033,5,0
2,quantile_median_gate,2808.2506,83.0712,-6.2782,4,1
3,friend_log_mixture,2814.5288,81.9263,0.0000,0,0
4,friend_log_gate_only,2816.9656,81.8066,2.4368,1,4
5,direct_log_mae,3054.6052,140.1496,240.0764,0,5


model,conditional_quantile_gate,conditional_quantile_mixture,direct_log_mae,friend_log_gate_only,friend_log_mixture,quantile_median_gate
fold,,,,,,
1,2713.17,2720.54,3029.84,2748.66,2742.14,2734.55
2,2876.03,2879.58,3002.99,2889.77,2885.28,2892.98
3,2692.12,2703.59,2861.47,2713.31,2712.03,2707.28
4,2819.43,2837.00,3218.29,2842.63,2848.23,2833.85
5,2848.33,2856.92,3160.43,2890.45,2884.95,2872.60


In [6]:
pooled_slices = error_slices.groupby(["model", "slice"], as_index=False).agg(
    rows=("rows", "sum"), sum_absolute_error=("sum_absolute_error", "sum"))
pooled_slices["MAE_thb"] = pooled_slices.sum_absolute_error / pooled_slices.rows
pooled_slices.to_csv(OUT / "pooled_error_slices.csv", index=False)
display(pooled_slices.pivot(index="model", columns="slice", values="MAE_thb").round(2))
quantile_rule_comparison = fold_scores.pivot(index="fold", columns="model", values="MAE_thb")
quantile_rule_comparison = quantile_rule_comparison[["quantile_median_gate", "conditional_quantile_gate"]].copy()
quantile_rule_comparison["delta_conditional_minus_positive_median"] = (
    quantile_rule_comparison.conditional_quantile_gate - quantile_rule_comparison.quantile_median_gate)
display(quantile_rule_comparison.round(2))
diagnostics = pd.DataFrame([{"fold": i, **json.loads((OUT / f"fold{i}_diagnostics.json").read_text())}
                            for i in range(1, 6)])
display(diagnostics)


slice,10k_to_50k,50k_plus,positive,positive_below_10k,zero
model,,,,,
conditional_quantile_gate,4244.47,18289.66,3661.91,469.06,1849.72
conditional_quantile_mixture,4220.50,18430.18,3684.51,473.10,1845.54
direct_log_mae,5337.97,20548.95,4293.96,690.36,1718.61
friend_log_gate_only,4410.45,18140.43,3624.91,427.50,1946.02
friend_log_mixture,4358.16,18294.04,3641.96,425.21,1922.58
quantile_median_gate,4367.18,18076.15,3611.22,428.71,1942.67


model,quantile_median_gate,conditional_quantile_gate,delta_conditional_minus_positive_median
fold,,,
1,2734.55,2713.17,-21.38
2,2892.98,2876.03,-16.95
3,2707.28,2692.12,-15.16
4,2833.85,2819.43,-14.41
5,2872.60,2848.33,-24.27


,fold,quantile_crossing_rows,active_q_below_grid,refitted_baseline_MAE,published_baseline_MAE_rounded,delta_from_published_rounded
0,1,337,41,2742.138234,2743.14,-1.001766
1,2,512,59,2885.283824,2886.62,-1.336176
2,3,305,50,2712.033952,2711.28,0.753952
3,4,372,36,2848.233300,2848.55,-0.316700
4,5,324,49,2884.954801,2885.25,-0.295199


## 6. Decision

Keep the original procedure unless a challenger reduces mean fold MAE by at least ฿10 and wins at least four folds. This is a prespecified practical filter, not a significance test. Previously explored development folds still make the comparison exploratory. Full-data refitting, holdout scoring, Test prediction and Kaggle assembly are separate next steps.

In [7]:
eligible = ranking.loc[
    ranking.delta_vs_friend_thb.le(-MIN_MEAN_GAIN_THB) & ranking.folds_better.ge(MIN_FOLD_WINS)]
selected_name = eligible.iloc[0].model if len(eligible) else "friend_log_mixture"
summary = {
    "selected_for_further_validation": selected_name,
    "baseline_mean_MAE_thb": reconstructed_mean,
    "published_friend_mean_MAE_thb": experiment.REFERENCE_MEAN_MAE,
    "baseline_exactly_reproduced": bool(abs(reconstructed_mean - experiment.REFERENCE_MEAN_MAE) < .02),
    "best_measured_model": ranking.iloc[0].model,
    "best_measured_mean_MAE_thb": float(ranking.iloc[0].mean_MAE_thb),
    "rows": len(oof), "folds": 5,
    "holdout_scored": False, "test_used": False, "kaggle_submitted": False,
    "note": "Reused development folds; no untouched estimate or official NMAE claim",
}
(OUT / "summary.json").write_text(json.dumps(summary, indent=2) + "\n")
print(json.dumps(summary, indent=2))


{
  "selected_for_further_validation": "conditional_quantile_gate",
  "baseline_mean_MAE_thb": 2814.5288220393245,
  "published_friend_mean_MAE_thb": 2814.9680123616345,
  "baseline_exactly_reproduced": false,
  "best_measured_model": "conditional_quantile_gate",
  "best_measured_mean_MAE_thb": 2789.815935343261,
  "rows": 83200,
  "folds": 5,
  "holdout_scored": false,
  "test_used": false,
  "kaggle_submitted": false,
  "note": "Reused development folds; no untouched estimate or official NMAE claim"
}


## 7. Reused Holdout Check After OOF Selection

The development comparison selected the conditional-quantile gate. Compare that fixed choice with a newly refitted source baseline on the existing 20% holdout once. The holdout was previously inspected by the friend, so this is a secondary check, not independent final confirmation. No parameters or thresholds are selected from these scores.

In [8]:
import joblib
from sklearn.metrics import mean_absolute_error
from Task3.models.conditional_quantile_model import ConditionalQuantileSpending
validation_path = OUT / "holdout_comparison.csv"
validation_model_path = OUT / "selected_development_model.joblib"
if selected_name == "conditional_quantile_gate":
    if validation_path.exists():
        assert validation_model_path.exists(), "Incomplete validation artifacts"
        holdout_comparison = pd.read_csv(validation_path)
        selected_development_model = joblib.load(validation_model_path)
    else:
        selected_development_model = ConditionalQuantileSpending(features).fit(dev[features], dev.spending_30d)
        hold_raw = raw.iloc[hold_pos]
        hold_features = experiment.add_spending_features(hold_raw)[features]
        baseline_components = {}
        for component in ["positive_log", "continuous_cat", "continuous_lgb"]:
            model = experiment.make_component(component, features)
            positive = dev.spending_30d.gt(0)
            if component == "positive_log":
                model.fit(dev.loc[positive, features], dev.loc[positive, "spending_30d"])
            else:
                model.fit(dev[features], dev.spending_30d)
            baseline_components[component] = model.predict(hold_features)
        probability = selected_development_model.gate_.predict_proba(hold_features)[:, 1]
        gated = np.where(probability >= .5, np.maximum(baseline_components["positive_log"], 0), 0)
        continuous = np.maximum(.4 * baseline_components["continuous_cat"] + .6 * baseline_components["continuous_lgb"], 0)
        hold_predictions = {"friend_log_mixture": .8 * gated + .2 * continuous,
                            "conditional_quantile_gate": selected_development_model.predict(hold_features)}
        holdout_comparison = pd.DataFrame([
            {"model": name, "MAE_thb": mean_absolute_error(hold_raw.spending_30d, pred),
             "rows": len(hold_raw)} for name, pred in hold_predictions.items()])
        holdout_predictions = pd.DataFrame({"raw_index": hold_pos, "player_id": hold_raw.player_id.to_numpy(),
                                            "true": hold_raw.spending_30d.to_numpy(), **hold_predictions})
        experiment.atomic_parquet(holdout_predictions, OUT / "holdout_predictions.parquet")
        joblib.dump(selected_development_model, validation_model_path)
        holdout_comparison.to_csv(validation_path, index=False)
    display(holdout_comparison.round(4))
    summary["holdout_scored"] = True
    summary["holdout_note"] = "Previously reused holdout; secondary check of the fixed OOF-selected procedure"
    summary["holdout_comparison"] = holdout_comparison.to_dict("records")
    (OUT / "summary.json").write_text(json.dumps(summary, indent=2) + "\n")
else:
    print("No conditional-quantile candidate passed the prespecified OOF filter; retain the original model")


,model,MAE_thb,rows
0,friend_log_mixture,2780.5097,20800
1,conditional_quantile_gate,2769.9707,20800


## 8. Full-Train Model & Test Predictions

Refit the fixed candidate on all 104,000 labeled rows only when it passed the OOF filter and the secondary holdout check improves on the refitted baseline. Test is read for prediction after selection. Save raw player IDs and row positions; assembling or replacing a combined Kaggle CSV remains separate.

In [9]:
if selected_name == "conditional_quantile_gate":
    hold_scores = holdout_comparison.set_index("model").MAE_thb
    secondary_check_passed = hold_scores["conditional_quantile_gate"] < hold_scores["friend_log_mixture"]
else:
    secondary_check_passed = False
if secondary_check_passed:
    full_model_path = OUT / "selected_full_model.joblib"
    full_card_path = OUT / "selected_full_model.json"
    assert full_model_path.exists() == full_card_path.exists(), "Incomplete full-data artifact"
    if full_model_path.exists():
        model_card = json.loads(full_card_path.read_text())
        assert model_card["settings_sha256"] == hashlib.sha256(settings_path.read_bytes()).hexdigest()
        final_model = joblib.load(full_model_path)
    else:
        final_model = ConditionalQuantileSpending(features).fit(raw, raw.spending_30d)
        model_card = {
            "model": "conditional_quantile_gate", "train_rows": len(raw), "features": features,
            "train_sha256": settings["train_sha256"],
            "settings_sha256": hashlib.sha256(settings_path.read_bytes()).hexdigest(),
            "predictor_implementation_sha256": hashlib.sha256((TASK / "models" / "conditional_quantile_model.py").read_bytes()).hexdigest(),
            "source_baseline": "friend build_model_v5/v6, source recipe reconstructed",
            "development_cv_mean_MAE_thb": float(ranking.loc[ranking.model.eq(selected_name), "mean_MAE_thb"].item()),
            "metric_note": "Development procedure estimate, not a score measured on the full fit or official Kaggle NMAE",
            "decision_rule": "p <= 0.5: zero; otherwise positive quantile q=1-0.5/p; finite grid/interpolation/sorted quantiles",
        }
        temporary = full_model_path.with_suffix(".partial.joblib")
        joblib.dump(final_model, temporary)
        temporary.replace(full_model_path)
        full_card_path.write_text(json.dumps(model_card, indent=2) + "\n")
    test_path = TASK / "data" / "raw" / "test.csv"
    test_raw = pd.read_csv(test_path)
    assert test_raw.columns.tolist() == raw.drop(columns="spending_30d").columns.tolist()
    assert test_raw.player_id.is_unique and len(test_raw) == 25889
    assert not set(test_raw.player_id).intersection(raw.player_id)
    test_prediction = final_model.predict(test_raw)
    assert np.isfinite(test_prediction).all() and (test_prediction >= 0).all()
    test_output = pd.DataFrame({"test_row_index": np.arange(len(test_raw)),
                               "player_id": test_raw.player_id.to_numpy(), "task3": test_prediction})
    experiment.atomic_parquet(test_output, OUT / "test_predictions.parquet")
    (OUT / "test_prediction_card.json").write_text(json.dumps({
        "test_sha256": hashlib.sha256(test_path.read_bytes()).hexdigest(),
        "model": selected_name, "rows": len(test_raw), "prediction": "nonnegative THB",
        "mapping": "Raw player_id and Test row position retained; no combined CSV generated"}, indent=2) + "\n")
    summary["test_used"] = True
    summary["test_use"] = "Final prediction only, after fixed-procedure evaluation"
    summary["full_data_model_saved"] = True
    (OUT / "summary.json").write_text(json.dumps(summary, indent=2) + "\n")
    display(test_output.head())
    print("Saved full-data model and", len(test_output), "Test predictions")
else:
    print("Secondary check did not improve: keep the previous submission and inspect the recorded tradeoff")


,test_row_index,player_id,task3
0,0,P030374,0.000000
1,1,P113216,2332.195747
2,2,P015351,0.000000
3,3,P095374,0.000000
4,4,P059369,0.000000


Saved full-data model and 25889 Test predictions
